# 🩻 Fine-tuning de Foundation Model para Radiologia
## Hands-on Vibe Coding — Sociedade Paulista de Radiologia 2026

---

### O que vamos fazer?

Um modelo pré-treinado no ImageNet sabe reconhecer gatos, carros e 998 outras coisas.  
Mas ele **não sabe distinguir Normal de Pneumonia** em um raio-X.

Neste hands-on você vai ver isso acontecer em tempo real:

| Etapa | O que acontece |
|---|---|
| **Antes** | Os embeddings do ImageNet misturam Normal e Pneumonia — o modelo não enxerga diferença |
| **Fine-tuning** | Você treina só a última camada com imagens radiológicas |
| **Depois** | Os embeddings se reorganizam: Normal e Pneumonia ficam em regiões separadas do espaço |
| **Grad-CAM** | O modelo revela exatamente onde ele "olha" para tomar a decisão |

---

### Dataset: PneumoniaMNIST

- **5.856 imagens** de raio-X de tórax (64×64 pixels)
- **2 classes:** Normal vs Pneumonia Bacteriana/Viral
- Download automático via `pip install medmnist` — sem login, sem cadastro
- Fonte: Guangzhou Women and Children's Medical Center / Kermany et al., Cell 2018
- Licença: CC BY 4.0

---

### Como usar este notebook

1. **Ative o GPU:** Menu `Ambiente de execução` → `Alterar tipo de hardware` → **T4 GPU** → Salvar
2. Execute as **células pré-preenchidas** (marcadas com `# PRÉ-PREENCHIDA`) sem alterar
3. Nas **células de vibe coding**, copie o Prompt Card e cole no **Gemini** (ícone ✦ ou `Ctrl+Shift+I`)
4. Cole o código gerado e execute

---

### ⏱️ Cronograma (60 minutos)

| | Bloco | Tempo |
|---|---|---|
| 🔧 | Setup + Dataset | 13 min |
| 📊 | Embeddings ANTES (t-SNE) | 7 min |
| 🏗️ | Construir e Treinar | 20 min |
| 📊 | Embeddings DEPOIS (t-SNE) | 5 min |
| 👁️ | Grad-CAM | 8 min |
| 💬 | Discussão | 7 min |

---
---
# 🔧 Setup — Execute Primeiro

⚠️ **Antes de executar:** confirme que o GPU T4 está ativo.  
Menu `Ambiente de execução` → `Alterar tipo de hardware` → T4 GPU → Salvar → Reconectar.

As duas células abaixo são **pré-preenchidas**. Execute na ordem sem alterar.

In [ ]:
# PRÉ-PREENCHIDA — Instalação e Seed de Reprodutibilidade
!pip install medmnist -q

import os, random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
import medmnist
from medmnist import PneumoniaMNIST

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'✅ SEED={SEED} definida')
print(f'📦 TensorFlow: {tf.__version__} | GPU disponível: {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'📦 medmnist: {medmnist.__version__}')

In [ ]:
# PRÉ-PREENCHIDA — Funções auxiliares (t-SNE, Grad-CAM, extração de embeddings)
# Estas funções serão chamadas nas células seguintes

# ─── Extração de embeddings ───────────────────────────────────────────────────
def extrair_embeddings(modelo_base, X, batch_size=64):
    """Extrai embeddings usando o modelo_base (sem a camada de classificação)."""
    embeddings = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        emb = modelo_base(batch, training=False)
        embeddings.append(emb.numpy())
    return np.concatenate(embeddings, axis=0)


# ─── Visualização t-SNE ───────────────────────────────────────────────────────
def plotar_tsne(emb_antes, emb_depois, labels, titulo_antes='Antes do Fine-tuning',
                titulo_depois='Após Fine-tuning'):
    """Plota t-SNE lado a lado: embeddings antes e depois."""
    cores = ['#2196F3', '#F44336']  # azul=Normal, vermelho=Pneumonia
    nomes = ['Normal', 'Pneumonia']

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Espaço de Embeddings — t-SNE', fontsize=15, fontweight='bold')

    for ax, emb, titulo in zip(axes, [emb_antes, emb_depois], [titulo_antes, titulo_depois]):
        tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=500)
        proj = tsne.fit_transform(emb)
        for idx, (nome, cor) in enumerate(zip(nomes, cores)):
            mask = labels == idx
            ax.scatter(proj[mask, 0], proj[mask, 1], c=cor, label=nome,
                       alpha=0.6, s=15, edgecolors='none')
        ax.set_title(titulo, fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        ax.spines[:].set_visible(False)

    plt.tight_layout()
    plt.show()


# ─── Grad-CAM ─────────────────────────────────────────────────────────────────
def grad_cam(modelo, img_array, layer_name=None):
    """Gera heatmap Grad-CAM para uma imagem (shape: 1, H, W, C)."""
    if layer_name is None:
        # Última camada convolucional do EfficientNetB0
        for layer in reversed(modelo.layers):
            if isinstance(layer, tf.keras.layers.Conv2D):
                layer_name = layer.name
                break
        if layer_name is None:
            # Procura dentro do modelo base
            for layer in reversed(modelo.layers):
                if hasattr(layer, 'layers'):
                    for sublayer in reversed(layer.layers):
                        if isinstance(sublayer, tf.keras.layers.Conv2D):
                            layer_name = sublayer.name
                            break
                if layer_name:
                    break

    grad_model = tf.keras.Model(
        inputs=modelo.inputs,
        outputs=[modelo.get_layer(layer_name).output, modelo.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap).numpy()
    heatmap = np.maximum(heatmap, 0) / (heatmap.max() + 1e-8)
    return heatmap


def plotar_gradcam(modelo, X_test, y_test, y_pred, y_proba, n_corretos=4, n_errados=4):
    """Visualiza Grad-CAM para n imagens corretas e n erradas."""
    label_names = {0: 'Normal', 1: 'Pneumonia'}
    corretos = np.where(y_pred == y_test)[0]
    errados  = np.where(y_pred != y_test)[0]
    rng = np.random.default_rng(SEED)
    idx_corretos = rng.choice(corretos, min(n_corretos, len(corretos)), replace=False)
    idx_errados  = rng.choice(errados,  min(n_errados,  len(errados)),  replace=False)
    indices = np.concatenate([idx_corretos, idx_errados])
    titulos_grupo = (['✅ CORRETO'] * len(idx_corretos) + ['❌ ERRO'] * len(idx_errados))

    fig, axes = plt.subplots(len(indices), 3, figsize=(9, len(indices) * 3))
    fig.suptitle('Grad-CAM — O que o modelo "olha"', fontsize=14, fontweight='bold')
    if len(indices) == 1:
        axes = axes[np.newaxis, :]

    # Última conv layer do backbone
    last_conv = None
    for layer in reversed(modelo.layers):
        if hasattr(layer, 'layers'):
            for sl in reversed(layer.layers):
                if isinstance(sl, tf.keras.layers.Conv2D):
                    last_conv = sl.name
                    break
        if last_conv:
            break

    for row, (idx, grupo) in enumerate(zip(indices, titulos_grupo)):
        img = X_test[idx]         # (64, 64, 3) float32
        img_batch = img[np.newaxis]
        real = label_names[y_test[idx]]
        pred = label_names[y_pred[idx]]
        conf = y_proba[idx] if y_pred[idx] == 1 else 1 - y_proba[idx]
        cor  = 'green' if grupo.startswith('✅') else 'red'

        # Imagem original
        axes[row, 0].imshow(img[:, :, 0], cmap='gray')
        axes[row, 0].set_title(f'{grupo}\nReal: {real} | Pred: {pred} ({conf:.0%})',
                               color=cor, fontsize=9)
        axes[row, 0].axis('off')

        # Heatmap Grad-CAM
        heatmap = grad_cam(modelo, img_batch, layer_name=last_conv)
        heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], [64, 64]).numpy().squeeze()
        axes[row, 1].imshow(heatmap_resized, cmap='jet')
        axes[row, 1].set_title('Grad-CAM (heatmap)', fontsize=9)
        axes[row, 1].axis('off')

        # Sobreposição
        axes[row, 2].imshow(img[:, :, 0], cmap='gray')
        axes[row, 2].imshow(heatmap_resized, cmap='jet', alpha=0.45)
        axes[row, 2].set_title('Sobreposição', fontsize=9)
        axes[row, 2].axis('off')

    plt.tight_layout()
    plt.show()


print('✅ Funções auxiliares carregadas: extrair_embeddings(), plotar_tsne(), plotar_gradcam()')

---
---
# 📋 Prompt Card 1 — Carregar e Visualizar o Dataset (8 min)

**Copie este prompt e cole no Gemini** (ícone ✦ à esquerda da célula ou `Ctrl+Shift+I`):

---
> Usando a biblioteca `medmnist` (já importada), carregue o `PneumoniaMNIST` com `download=True` e `size=64` para os splits de treino, validação e teste. Em seguida:
> 1. Converta as imagens para float32 no intervalo [0, 1], shape `(N, 64, 64, 3)` — repita o canal cinza 3 vezes usando `np.repeat` (necessário para o EfficientNetB0).
> 2. Converta os labels para arrays 1D de inteiros.
> 3. Imprima shape e distribuição de classes (Normal vs Pneumonia) de cada split.
> 4. Plote uma grade 4×4 com 16 imagens aleatórias do treino. Use `cmap='gray'` e coloque o label (Normal/Pneumonia) como título de cada imagem. Use `np.random.default_rng(42)` para selecionar as imagens.

---
**Cole o código gerado abaixo e execute:**

In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini (Prompt Card 1)

---
---
# 📊 Embeddings ANTES do Fine-tuning — Execute e Observe (7 min)

Esta célula é **pré-preenchida**. Ela usa o EfficientNetB0 com pesos do ImageNet (sem fine-tuning)
para extrair embeddings das imagens de teste e visualiza com t-SNE.

> **O que esperar:** Normal e Pneumonia aparecerão **misturados** — o modelo ImageNet não distingue essas categorias clínicas.

In [ ]:
# PRÉ-PREENCHIDA — Embeddings ANTES do fine-tuning
print('⏳ Carregando EfficientNetB0 com pesos ImageNet...')
efficientnet_base = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(64, 64, 3),
    pooling='avg'
)
efficientnet_base.trainable = False

print('⏳ Extraindo embeddings do conjunto de teste (ImageNet puro)...')
# Pré-processa para o EfficientNet (espera valores [0,255] internamente)
X_test_prep = tf.keras.applications.efficientnet.preprocess_input(X_test * 255.0)
emb_antes = extrair_embeddings(efficientnet_base, X_test_prep)

print(f'Shape dos embeddings: {emb_antes.shape}')
print('⏳ Calculando t-SNE (pode levar ~30 segundos)...')

plotar_tsne(
    emb_antes, emb_antes,
    y_test,
    titulo_antes='ANTES — EfficientNetB0 (ImageNet)',
    titulo_depois='← execute este lado após o treino'
)

print('\n💡 Observe: Normal (azul) e Pneumonia (vermelho) estão misturados.')
print('   O modelo ImageNet não aprendeu o que distingue raios-X clínicos.')

---
---
# 📋 Prompt Card 2 — Construir o Modelo com EfficientNetB0 (8 min)

**Copie este prompt e cole no Gemini:**

---
> Usando TensorFlow/Keras e a variável `efficientnet_base` já carregada (EfficientNetB0 sem topo, pesos ImageNet, pooling='avg', trainable=False), construa um modelo Sequential para classificação binária de imagens 64×64×3:
> 1. Use `efficientnet_base` como primeira camada (base congelada — não altere o trainable).
> 2. Adicione BatchNormalization.
> 3. Adicione Dense(256, activation='relu').
> 4. Adicione Dropout(0.4, seed=42).
> 5. Adicione Dense(1, activation='sigmoid') — classificação binária.
> 6. Compile com Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')].
> 7. Mostre o summary do modelo. Imprima também quantos parâmetros são treináveis vs congelados.

---
**Cole o código gerado abaixo e execute:**

In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini (Prompt Card 2)

---
---
# 📋 Prompt Card 3 — Treinar o Modelo (12 min)

**Copie este prompt e cole no Gemini:**

---
> Treine o modelo criado anteriormente usando `X_train`, `y_train`, `X_val`, `y_val`. Use:
> - `epochs=25`, `batch_size=64`
> - Pré-processe as entradas com `tf.keras.applications.efficientnet.preprocess_input(X * 255.0)` dentro de uma função de treino ou passando os dados já pré-processados.
> - Callbacks: `EarlyStopping(monitor='val_auc', patience=6, restore_best_weights=True, mode='max')` e `ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)`.
> - Após o treino, plote dois gráficos lado a lado: curva de loss (treino vs validação) e curva de AUC (treino vs validação) por época.
> - Imprima a AUC final no conjunto de validação.

---
**Cole o código gerado abaixo e execute:**

> ⏳ Com GPU T4: ~3–5 min. Sem GPU: ~15–20 min.

In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini (Prompt Card 3)

---
---
# 📊 Embeddings DEPOIS do Fine-tuning — O Momento Central (5 min)

Esta célula é **pré-preenchida**. Agora extraímos embeddings com o modelo **após** o fine-tuning
e comparamos lado a lado com o estado original.

> **O que esperar:** Normal e Pneumonia agora aparecerão **separados** em regiões distintas.

In [ ]:
# PRÉ-PREENCHIDA — Embeddings DEPOIS do fine-tuning + comparação t-SNE

# Extrai embeddings usando a base fine-tunada (sem a Dense head)
modelo_sem_cabeca = tf.keras.Model(
    inputs=modelo.input,
    outputs=modelo.layers[-3].output  # saída do BatchNorm, antes do Dense final
)

print('⏳ Extraindo embeddings após fine-tuning...')
X_test_prep = tf.keras.applications.efficientnet.preprocess_input(X_test * 255.0)
emb_depois = extrair_embeddings(modelo_sem_cabeca, X_test_prep)

print(f'Shape embeddings após fine-tuning: {emb_depois.shape}')
print('⏳ Calculando t-SNE para ambos os espaços...')

plotar_tsne(
    emb_antes, emb_depois,
    y_test,
    titulo_antes='ANTES — EfficientNetB0 (ImageNet puro)',
    titulo_depois='DEPOIS — Fine-tuned em RX de Tórax'
)

print('\n💡 O fine-tuning reorganizou o espaço de embeddings:')
print('   Normal (azul) e Pneumonia (vermelho) formam grupos separados.')
print('   O modelo aprendeu o que é clinicamente relevante nas imagens radiológicas.')

---
---
# 📊 Avaliação no Conjunto de Teste — Pré-preenchida

Execute para obter AUC-ROC, matriz de confusão, sensibilidade e especificidade.

In [ ]:
# PRÉ-PREENCHIDA — Avaliação completa no conjunto de teste
X_test_prep_eval = tf.keras.applications.efficientnet.preprocess_input(X_test * 255.0)
y_proba = modelo.predict(X_test_prep_eval, verbose=0).ravel()

fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

# Threshold ótimo: índice de Youden
youden_idx = np.argmax(tpr - fpr)
best_threshold = thresholds[youden_idx]
y_pred = (y_proba >= best_threshold).astype(int)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity  = tp / (tp + fn)
specificity  = tn / (tn + fp)
accuracy     = (tp + tn) / len(y_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(fpr, tpr, color='blue', lw=2, label=f'AUC = {auc_score:.3f}')
ax1.scatter(fpr[youden_idx], tpr[youden_idx], color='red', s=100, zorder=5,
            label=f'Threshold ótimo = {best_threshold:.2f}')
ax1.plot([0, 1], [0, 1], 'k--')
ax1.set_xlabel('1 - Especificidade (FPR)')
ax1.set_ylabel('Sensibilidade (TPR)')
ax1.set_title('Curva ROC — PneumoniaMNIST (Teste)')
ax1.legend(); ax1.grid(True)

disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Pneumonia'])
disp.plot(ax=ax2, colorbar=False, cmap='Blues')
ax2.set_title('Matriz de Confusão — Teste')

plt.tight_layout()
plt.show()

print(f'\n📊 Resultados no Conjunto de Teste:')
print(f'   AUC-ROC       : {auc_score:.3f}')
print(f'   Threshold ótimo: {best_threshold:.3f}')
print(f'   Acurácia       : {accuracy:.3f}')
print(f'   Sensibilidade  : {sensitivity:.3f}  (recall de Pneumonia)')
print(f'   Especificidade : {specificity:.3f}')

---
---
# 📋 Prompt Card 4 — Grad-CAM: O que o modelo "olha"? (8 min)

A função `plotar_gradcam()` já está pronta (célula de setup). Você só precisa chamá-la.

**Copie este prompt e cole no Gemini:**

---
> Chame a função `plotar_gradcam()` que já está definida, passando os seguintes argumentos:
> - `modelo`: o modelo treinado
> - `X_test`: o array de imagens de teste (já pré-processadas como float [0,1] com 3 canais)
> - `y_test`: os labels reais
> - `y_pred`: as predições binárias já calculadas
> - `y_proba`: as probabilidades brutas do modelo
> - `n_corretos=3, n_errados=3`
>
> Antes de chamar a função, pré-processe X_test com `tf.keras.applications.efficientnet.preprocess_input(X_test * 255.0)` e salve em uma variável `X_test_prep_gcam`. Passe `X_test_prep_gcam` como o argumento `X_test` da função.

---
**Cole o código gerado abaixo e execute:**

> ⏳ Cada Grad-CAM leva ~2s. Para 6 imagens: ~12 segundos total.

In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini (Prompt Card 4)

---
---
# 💬 Discussão Clínica (7 min)

## O que aconteceu?

| | Antes | Depois |
|---|---|---|
| **Parâmetros treinados** | 0 (congelado) | ~262K (só a head) |
| **Conhecimento** | ImageNet: gatos, carros... | RX tórax: Normal vs Pneumonia |
| **t-SNE** | Classes misturadas | Classes separadas |
| **AUC no teste** | ~0.50 (acaso) | ~0.90–0.95 |

## Perguntas para reflexão

**1. Por que a base fica congelada?**
> O EfficientNetB0 tem 4,05M parâmetros que já aprenderam a detectar bordas, texturas, gradientes. Esses detectores de baixo nível funcionam bem em radiologia. Retreinar tudo com 4.708 imagens causaria *catastrophic forgetting*.

**2. O Grad-CAM está olhando para onde você esperava?**
> Nos casos corretos de Pneumonia: o heatmap deve destacar regiões de consolidação/opacidade.  
> Nos erros: o heatmap pode estar focando em artefatos ou regiões irrelevantes — **isso é explicabilidade real**.

**3. Por que AUC e não acurácia?**
> No dataset: 1.214 Normal vs 3.494 Pneumonia (treino). Um modelo que diz "sempre Pneumonia" teria ~74% de acurácia mas AUC = 0.5.

**4. Limitações para uso clínico real:**
- Imagens 64×64 perdem detalhes diagnósticos críticos
- Dataset: crianças de Guangzhou — generalização para população brasileira adulta? 
- Sem validação prospectiva
- Pneumonia bacteriana + viral agrupadas (sem distinção)

**5. O que este experimento demonstra sobre IA em radiologia?**
> Modelos de propósito geral (ImageNet) podem ser especializados para tarefas clínicas com datasets relativamente pequenos. O conhecimento aprendido em imagens naturais transfere parcialmente para imagens médicas — especialmente em texturas e padrões de baixo nível.

---

## Referências

- **PneumoniaMNIST:** Kermany DS et al., *Cell*, 2018. doi: 10.1016/j.cell.2018.02.010  
- **MedMNIST v2:** Yang J et al., *Scientific Data*, 2023. doi: 10.1038/s41597-022-01721-8  
- **EfficientNet:** Tan M, Le QV, *ICML*, 2019. arXiv: 1905.11946  
- **Grad-CAM:** Selvaraju RR et al., *ICCV*, 2017. arXiv: 1610.02391

---
---
---
# 📚 GABARITO — Código de Referência (Apenas para o Instrutor)

> Implementação completa de referência.  
> Use apenas se algum participante travar completamente.

In [ ]:
# GABARITO — Prompt Card 1: Carregar e visualizar PneumoniaMNIST
train_ds = PneumoniaMNIST(split='train', download=True, size=64)
val_ds   = PneumoniaMNIST(split='val',   download=True, size=64)
test_ds  = PneumoniaMNIST(split='test',  download=True, size=64)

def prepare_split(dataset):
    imgs   = np.stack([np.array(dataset[i][0]) for i in range(len(dataset))]).astype('float32') / 255.0
    labels = np.array([int(dataset[i][1]) for i in range(len(dataset))])
    if imgs.ndim == 3:  # (N, H, W) → (N, H, W, 1)
        imgs = imgs[..., np.newaxis]
    if imgs.shape[-1] == 1:  # grayscale → RGB sintético
        imgs = np.repeat(imgs, 3, axis=-1)
    return imgs, labels

X_train, y_train = prepare_split(train_ds)
X_val,   y_val   = prepare_split(val_ds)
X_test,  y_test  = prepare_split(test_ds)

label_names = {0: 'Normal', 1: 'Pneumonia'}
for name, X, y in [('Treino', X_train, y_train), ('Validação', X_val, y_val), ('Teste', X_test, y_test)]:
    print(f'{name}: shape={X.shape} | Normal={int((y==0).sum())} | Pneumonia={int((y==1).sum())}')

rng = np.random.default_rng(SEED)
indices = rng.choice(len(train_ds), 16, replace=False)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('PneumoniaMNIST — Amostras do Conjunto de Treino', fontsize=13, fontweight='bold')
for ax, idx in zip(axes.flat, indices):
    img, label = train_ds[idx]
    ax.imshow(np.array(img).squeeze(), cmap='gray')
    lbl = int(label)
    ax.set_title(label_names[lbl], color='red' if lbl == 1 else 'green', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# GABARITO — Prompt Card 2: Construir modelo com EfficientNetB0
efficientnet_base = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(64, 64, 3),
    pooling='avg'
)
efficientnet_base.trainable = False

modelo = tf.keras.Sequential([
    efficientnet_base,
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.4, seed=SEED),
    tf.keras.layers.Dense(1, activation='sigmoid'),
], name='PneumoniaCNN_EfficientNet')

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
modelo.build((None, 64, 64, 3))
modelo.summary()

total     = sum(np.prod(v.shape) for v in modelo.trainable_variables)
congelado = sum(np.prod(v.shape) for v in modelo.non_trainable_variables)
print(f'\n  Parâmetros treináveis  : {total:,}')
print(f'  Parâmetros congelados  : {congelado:,}')

In [ ]:
# GABARITO — Prompt Card 3: Treinar o modelo
X_train_prep = tf.keras.applications.efficientnet.preprocess_input(X_train * 255.0)
X_val_prep   = tf.keras.applications.efficientnet.preprocess_input(X_val   * 255.0)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=6, restore_best_weights=True, mode='max'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6
    ),
]

history = modelo.fit(
    X_train_prep, y_train,
    epochs=25,
    batch_size=64,
    validation_data=(X_val_prep, y_val),
    callbacks=callbacks,
    verbose=1
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'],     label='Treino')
ax1.plot(history.history['val_loss'], label='Validação')
ax1.set_title('Loss por Época'); ax1.set_xlabel('Época'); ax1.legend(); ax1.grid(True)

ax2.plot(history.history['auc'],     label='Treino')
ax2.plot(history.history['val_auc'], label='Validação')
ax2.set_title('AUC por Época'); ax2.set_xlabel('Época'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.show()

best_val_auc = max(history.history['val_auc'])
print(f'\n✅ Melhor AUC na validação: {best_val_auc:.4f}')

In [ ]:
# GABARITO — Prompt Card 4: Grad-CAM
X_test_prep_gcam = tf.keras.applications.efficientnet.preprocess_input(X_test * 255.0)
plotar_gradcam(modelo, X_test_prep_gcam, y_test, y_pred, y_proba, n_corretos=3, n_errados=3)

# 🏥 Construa seu próprio Assistente de Laudos com IA
## Hands-on Vibe Coding — Sociedade Paulista de Radiologia 2026

---

**Você vai sair daqui com um app funcional rodando no browser, construído por você com ajuda do Gemini — sem instalar nada, sem saber programar.**

O que vamos construir juntos:

| Iteração | O que o app faz |
|---|---|
| ▶ Versão 1 | Recebe achados em texto livre e gera laudo estruturado |
| ▶ Versão 2 | Adiciona seleção de modalidade e região anatômica |
| ▶ Versão 3 | Layout profissional em duas colunas + botão copiar |
| ▶ Versão 4 | Alerta automático de achados críticos |
| ★ Livre | Você especializa para sua subespecialidade |

---

### ✅ O que você precisa

- Conta Google (para o Colab) — já tem
- API Key do Gemini — **gratuita em [aistudio.google.com](https://aistudio.google.com)** (pegar agora, leva 1 minuto)
- Browser atualizado — já tem
- Nada mais

---

### 🎯 O que é Vibe Coding?

> **Você descreve em português o que quer que o código faça. O Gemini escreve o código. Você executa e vê o resultado.**
>
> Regra de ouro: **você nunca edita o código manualmente**. Se algo não ficou como queria, você pede ao Gemini para corrigir, descrevendo o problema.

---

### ⏱️ Cronograma (90 minutos)

| Bloco | Conteúdo | Tempo |
|---|---|---|
| 1 | Contexto + Setup + API Key | 15 min |
| 2 | Primeiro prompt → primeiro app no ar | 20 min |
| 3 | Iterando: 3 melhorias guiadas | 30 min |
| 4 | Personalização livre | 15 min |
| 5 | Demo + discussão clínica | 10 min |

---
---
# 🔧 Bloco 1 — Setup (15 min)

## Passo 1: Pegar sua API Key gratuita

1. Acesse **[aistudio.google.com](https://aistudio.google.com)** (use a mesma conta Google deste Colab)
2. Clique em **"Get API key"** → **"Create API key"**
3. Copie a chave gerada (começa com `AIza...`)
4. No Colab, clique no ícone 🔑 **Secrets** na barra lateral esquerda
5. Clique em **"+ Adicionar novo secret"**, nome: `GEMINI_API_KEY`, valor: cole a chave
6. Ative o toggle "Acesso do notebook"

> 💡 **Por que Secrets e não colar direto?** Para nunca expor sua API key se você compartilhar o notebook.

## Passo 2: Instalar as bibliotecas

Execute a célula abaixo (já preenchida). Ela instala:
- **`google-generativeai`** — SDK para chamar o Gemini
- **`gradio`** — cria interfaces web com poucas linhas de Python

In [ ]:
# ============================================================
# CÉLULA PRÉ-PREENCHIDA — Execute sem alterar
# ============================================================
!pip install google-generativeai gradio -q

import google.generativeai as genai
import gradio as gr
from google.colab import userdata

print("✅ Bibliotecas instaladas com sucesso!")
print(f"   gradio version: {gr.__version__}")
print(f"   google-generativeai version: {genai.__version__}")

## Passo 3: Configurar a API Key e testar o Gemini

Execute a célula abaixo. Ela lê a chave do Secrets e faz um teste rápido.

In [ ]:
# ============================================================
# CÉLULA PRÉ-PREENCHIDA — Execute sem alterar
# ============================================================
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)
    modelo = genai.GenerativeModel('gemini-1.5-flash')

    # Teste rápido
    resposta_teste = modelo.generate_content(
        "Em uma frase, o que é radiologia diagnóstica? Responda em português."
    )
    print("✅ API Key configurada e Gemini respondendo!")
    print(f"\n🤖 Gemini diz: {resposta_teste.text}")

except Exception as e:
    print(f"❌ Erro: {e}")
    print("Verifique se criou o Secret com o nome exato: GEMINI_API_KEY")
    print("E se ativou o toggle 'Acesso do notebook'.")

---
---
# ✦ Bloco 2 — Primeiro Prompt, Primeiro App (20 min)

Agora começa o vibe coding. Você vai usar o **Gemini do Colab** para escrever o código.

**Como abrir o Gemini no Colab:**
- Clique no ícone ✦ que aparece à esquerda de qualquer célula de código, OU
- Use o atalho `Ctrl + Shift + I`

---

## 📋 Prompt Card 1 — App Básico

**Copie exatamente este prompt e cole no Gemini do Colab:**

---
> Usando Python com as bibliotecas `gradio` e `google.generativeai` (já importadas) e a variável `modelo` (já configurada com gemini-1.5-flash), crie uma interface web simples de assistente de laudos radiológicos. A interface deve ter: (1) uma caixa de texto grande para o médico digitar os achados em texto livre, com placeholder de exemplo; (2) um botão "Gerar Laudo"; (3) uma área de saída mostrando o laudo. O prompt para o Gemini deve instruí-lo a agir como radiologista experiente e estruturar o laudo com três seções em Markdown: ACHADOS (organizados), DIAGNÓSTICOS DIFERENCIAIS (3-5 hipóteses com probabilidade em %), IMPRESSÃO DIAGNÓSTICA (conclusão objetiva). Use `gr.Interface` simples. No final use `app.launch(share=True)` para gerar um link público.
---

**Cole o código gerado na célula abaixo e execute.**

> ⏳ Aguarde aparecer uma URL pública tipo `https://xxxx.gradio.live` — é o seu app no ar!

In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini (Prompt Card 1)

---
---
# 🔄 Bloco 3 — Iterando com Vibe Coding (30 min)

**Regra:** você não edita o código da célula anterior. Você pede ao Gemini para criar uma **versão melhorada** e cola em uma nova célula.

> 💡 O Gemini do Colab tem contexto das células executadas. Não precisa re-explicar o que já existe.

---

## 📋 Prompt Card 2 — Modalidade e Região Anatômica

**Copie exatamente este prompt e cole no Gemini:**

---
> Reescreva o assistente de laudos com estas melhorias: (1) adicione um dropdown "Modalidade" com as opções: Raio-X, Tomografia Computadorizada, Ressonância Magnética, Ultrassonografia, Mamografia, PET-CT; (2) adicione um dropdown "Região Anatômica" com: Tórax, Abdome, Encéfalo, Coluna Vertebral, Mama, Musculoesquelético, Pelve, Pescoço; (3) passe modalidade e região para o prompt do Gemini para que o laudo inclua também a técnica padrão do exame e a seção "Conduta Sugerida"; (4) mostre a saída como Markdown formatado usando `gr.Markdown` ao invés de Textbox. Mantenha o `app.launch(share=True)` no final.
---

**Cole o código gerado na célula abaixo e execute.**

In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini (Prompt Card 2)

---

## 📋 Prompt Card 3 — Layout Profissional em Duas Colunas

**Copie exatamente este prompt e cole no Gemini:**

---
> Refatore o app anterior para usar `gr.Blocks` com layout em duas colunas: coluna esquerda (campos de entrada: modalidade, região, caixa de achados e botão "📝 Gerar Laudo"), coluna direita (laudo em Markdown). Adicione no topo um título estilizado com `gr.Markdown` incluindo o nome do seu serviço/hospital. Adicione também um tema visual com `gr.themes.Soft()`. Mantenha o `app.launch(share=True)` no final.
---

**Cole o código gerado na célula abaixo e execute.**

In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini (Prompt Card 3)

---

## 📋 Prompt Card 4 — Alerta de Achados Críticos

**Copie exatamente este prompt e cole no Gemini:**

---
> Adicione ao app uma segunda funcionalidade: um botão "⚠️ Verificar Achados Críticos" que envia os mesmos achados ao Gemini com o seguinte prompt específico: "Você é um radiologista. Analise estes achados de [modalidade] de [região]: [achados]. Existem achados críticos que requerem comunicação imediata (emergência radiológica)? Se SIM: liste em ordem de prioridade começando com '⚠️ ACHADOS CRÍTICOS:' e indique a ação urgente. Se NÃO: responda apenas '✅ Sem achados críticos. Comunicação de rotina adequada.' Seja direto e objetivo." Mostre o resultado abaixo do laudo principal em uma caixa separada com label "Verificação de Achados Críticos". Mantenha o botão de gerar laudo funcionando. Mantenha o `app.launch(share=True)`.
---

**Cole o código gerado na célula abaixo e execute.**

In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini (Prompt Card 4)

---
---
# ★ Bloco 4 — Personalização Livre (15 min)

Agora é com você. Especialize o assistente para **sua** subespecialidade.

## 📋 Prompt Card 5 — Especialização

**Escolha UMA das sugestões abaixo ou escreva a sua própria:**

---
**Se você faz TC de Tórax:**
> Modifique o prompt do Gemini para especializar o assistente em TC de Tórax. O laudo deve avaliar sistematicamente: parênquima pulmonar (descreva densidade, distribuição, padrão — vidro fosco, consolidação, nódulos), estruturas vasculares (calibre da aorta e artéria pulmonar), mediastino, pleura e parede torácica. Se houver nódulos, mencione critérios de Lung-RADS. Se houver achados vasculares, mencione TEP no diferencial quando pertinente. Ajuste os dropdowns para focar em TC de Tórax.

**Se você faz RM de Encéfalo:**
> Modifique o prompt do Gemini para especializar em RM de Encéfalo. O laudo deve avaliar sistematicamente: parênquima (sinal em T1/T2/FLAIR, restrição à difusão, realce pelo contraste), ventrículos e espaços subaracnóideos, estruturas da fossa posterior, vascular. No diferencial de lesões encefálicas, sempre considere: neoplasia primária, metástase, desmielinizante, infecciosa e vascular. Ajuste os dropdowns para RM de Encéfalo.

**Se você faz Mamografia:**
> Modifique o prompt do Gemini para especializar em Mamografia e Ultrassom de Mama. O laudo deve descrever: densidade mamária (A, B, C ou D do BI-RADS), achados (calcificações, nódulos, assimetrias, distorções), e concluir com a categoria BI-RADS (0 a 6) com a recomendação de conduta padrão para cada categoria. Ajuste os dropdowns para mamografia e US de mama.

**Personalização livre:**
> Modifique o assistente para [DESCREVA SUA SUBESPECIALIDADE]. O laudo deve avaliar: [DESCREVA OS ITENS OBRIGATÓRIOS DO SEU LAUDO]. Ajuste os dropdowns de modalidade e região para o seu contexto.
---

**Cole o código gerado na célula abaixo e execute.**

In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini (Prompt Card 5 — sua especialidade)

---

## 💡 Terminou cedo? Tente estes prompts bônus

**Bônus A — Histórico de laudos na sessão:**
> Adicione ao app uma aba "Histórico" que salva numa lista todos os laudos gerados durante a sessão (achados + laudo + timestamp). Mostre o histórico como tabela usando `gr.DataFrame`. Adicione um botão para limpar o histórico.

**Bônus B — Comparação de modelos:**
> Modifique o app para gerar o laudo com dois modelos em paralelo: `gemini-1.5-flash` (rápido) e `gemini-1.5-pro` (mais detalhado). Mostre os dois laudos lado a lado para comparação.

**Bônus C — Exportar em PDF:**
> Adicione um botão "Exportar PDF" que converte o laudo gerado para um arquivo PDF e oferece download. Use a biblioteca `fpdf2` (instale com pip).

---
---
# 💬 Bloco 5 — Discussão (10 min)

## 2-3 voluntários: mostrem o que construíram

---

## Onde isso já está sendo usado?

- **Nuance PowerScribe** / **Philips Radiology Operations Command Center** — integração de LLM em RIS/PACS
- **Aidoc**, **RapidAI**, **Viz.ai** — triagem de achados críticos em tempo real
- **Google Health** + **DeepMind** — assistentes de laudo em fase de validação clínica
- **Hospitais brasileiros** — ainda em fase piloto, mas crescendo

## Limitações importantes (discussão clínica)

| Limitação | Implicação |
|---|---|
| O Gemini não vê a imagem | Ele só estrutura o que você descreve — o raciocínio clínico ainda é seu |
| Alucinação | Pode gerar diagnósticos diferenciais plausíveis mas errados — sempre revise |
| Sem validação clínica | Este app é um **protótipo educacional**, não um dispositivo médico |
| LGPD / HIPAA | Não inserir dados de pacientes reais em APIs externas |
| Latência | Depende de conexão e da cota gratuita (15 req/min no free tier) |

## O que você demonstrou hoje

> Um radiologista sem experiência em programação pode prototipar uma ideia clínica funcional em menos de 1 hora. O que antes exigia um time de TI agora exige apenas clareza para descrever o problema.

---

**Obrigado pela participação! Dúvidas e ideias: fique após a sessão.**

---
---
---
# 📚 GABARITO — Código de Referência (Apenas para o Instrutor)

> Use apenas se algum participante travar completamente.  
> O objetivo é que todos construam via Gemini, não usando este código.

In [ ]:
# ============================================================
# GABARITO — Versão 1: App básico (resultado do Prompt Card 1)
# ============================================================
import google.generativeai as genai
import gradio as gr
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
modelo = genai.GenerativeModel('gemini-1.5-flash')


def analisar_laudo_v1(achados):
    if not achados.strip():
        return "⚠️ Por favor, insira os achados radiológicos."

    prompt = f"""Você é um radiologista experiente. Analise os seguintes achados radiológicos
e gere um laudo estruturado em português brasileiro.

ACHADOS DESCRITOS PELO MÉDICO:
{achados}

Gere um laudo com EXATAMENTE esta estrutura em Markdown:

## ACHADOS
[Organize e estruture os achados descritos]

## DIAGNÓSTICOS DIFERENCIAIS
[Liste 3-5 hipóteses diagnósticas com probabilidade estimada em %]

## IMPRESSÃO DIAGNÓSTICA
[Conclusão objetiva com a hipótese principal]

Use terminologia médica adequada. Seja preciso e objetivo."""

    resposta = modelo.generate_content(prompt)
    return resposta.text


app_v1 = gr.Interface(
    fn=analisar_laudo_v1,
    inputs=gr.Textbox(
        label="Achados Radiológicos",
        placeholder="Descreva os achados como você ditaria...\n\nExemplo: opacidade em vidro fosco bilateral nos lobos superiores, com predomínio periférico, sem derrame pleural...",
        lines=6
    ),
    outputs=gr.Textbox(label="Laudo Gerado", lines=15),
    title="🏥 Assistente de Laudos Radiológicos",
    description="Digite os achados em texto livre e o Gemini estrutura o laudo automaticamente."
)

app_v1.launch(share=True)

In [ ]:
# ============================================================
# GABARITO — Versão 2: Com modalidade + região + Markdown
# (resultado dos Prompt Cards 2 e 3 combinados)
# ============================================================
MODALIDADES = [
    "Raio-X", "Tomografia Computadorizada", "Ressonância Magnética",
    "Ultrassonografia", "Mamografia", "PET-CT"
]
REGIOES = [
    "Tórax", "Abdome", "Encéfalo", "Coluna Vertebral",
    "Mama", "Musculoesquelético", "Pelve", "Pescoço"
]


def analisar_laudo_v2(achados, modalidade, regiao):
    if not achados.strip():
        return "⚠️ Por favor, insira os achados radiológicos."

    prompt = f"""Você é um radiologista experiente especialista em {modalidade} de {regiao}.
Analise os seguintes achados e gere um laudo estruturado em português brasileiro.

EXAME: {modalidade} de {regiao}
ACHADOS DESCRITOS:
{achados}

## 📋 TÉCNICA
[Descreva a técnica padrão para {modalidade} de {regiao}]

## 🔍 ACHADOS
[Organize e estruture os achados descritos]

## 🩺 DIAGNÓSTICOS DIFERENCIAIS
[3-5 hipóteses com probabilidade estimada em %]

## 💡 IMPRESSÃO DIAGNÓSTICA
[Conclusão objetiva]

## 📌 CONDUTA SUGERIDA
[Complementação diagnóstica ou seguimento, se pertinente]"""

    resposta = modelo.generate_content(prompt)
    return resposta.text


with gr.Blocks(title="Assistente de Laudos", theme=gr.themes.Soft()) as app_v2:
    gr.Markdown("# 🏥 Assistente de Laudos Radiológicos\n*Powered by Gemini AI — SPR 2026*")

    with gr.Row():
        with gr.Column():
            modalidade = gr.Dropdown(
                choices=MODALIDADES, value="Tomografia Computadorizada",
                label="Modalidade"
            )
            regiao = gr.Dropdown(
                choices=REGIOES, value="Tórax",
                label="Região Anatômica"
            )
            achados = gr.Textbox(
                label="Achados (texto livre)",
                placeholder="Descreva os achados como você ditaria...",
                lines=7
            )
            btn = gr.Button("📝 Gerar Laudo", variant="primary", size="lg")

        with gr.Column():
            laudo = gr.Markdown(label="Laudo Gerado")

    btn.click(analisar_laudo_v2, inputs=[achados, modalidade, regiao], outputs=laudo)

app_v2.launch(share=True)

In [ ]:
# ============================================================
# GABARITO — Versão 3: Completa com achados críticos
# (resultado do Prompt Card 4)
# ============================================================

def analisar_completo(achados, modalidade, regiao):
    if not achados.strip():
        return "⚠️ Por favor, insira os achados radiológicos.", ""

    # Laudo estruturado
    prompt_laudo = f"""Você é um radiologista experiente especialista em {modalidade} de {regiao}.
Analise os seguintes achados e gere um laudo estruturado em português brasileiro.

EXAME: {modalidade} de {regiao}
ACHADOS DESCRITOS:
{achados}

## 📋 TÉCNICA
[Técnica padrão]

## 🔍 ACHADOS
[Achados organizados]

## 🩺 DIAGNÓSTICOS DIFERENCIAIS
[3-5 hipóteses com probabilidade em %]

## 💡 IMPRESSÃO DIAGNÓSTICA
[Conclusão objetiva]

## 📌 CONDUTA SUGERIDA
[Complementação ou seguimento]"""

    # Verificação de achados críticos
    prompt_criticos = f"""Você é um radiologista. Analise estes achados de {modalidade} de {regiao}:
{achados}

Existem achados críticos que requerem comunicação IMEDIATA (emergência radiológica)?
Se SIM: liste em ordem de prioridade começando com '⚠️ ACHADOS CRÍTICOS:' e indique a ação urgente.
Se NÃO: responda apenas '✅ Sem achados críticos. Comunicação de rotina adequada.'
Seja direto e objetivo."""

    resposta_laudo = modelo.generate_content(prompt_laudo)
    resposta_criticos = modelo.generate_content(prompt_criticos)

    return resposta_laudo.text, resposta_criticos.text


with gr.Blocks(title="Assistente de Laudos — SPR 2026", theme=gr.themes.Soft()) as app_v3:
    gr.Markdown(
        """# 🏥 Assistente de Laudos Radiológicos
        *Powered by Gemini AI — SPR 2026 | Protótipo Educacional*"""
    )

    with gr.Row():
        with gr.Column(scale=1):
            modalidade = gr.Dropdown(
                choices=MODALIDADES, value="Tomografia Computadorizada",
                label="Modalidade"
            )
            regiao = gr.Dropdown(
                choices=REGIOES, value="Tórax",
                label="Região Anatômica"
            )
            achados = gr.Textbox(
                label="Achados (texto livre)",
                placeholder="Descreva os achados como você ditaria...\n\nExemplo: opacidade em vidro fosco bilateral, periférica, lobos superiores, sem derrame pleural, traqueia centrada...",
                lines=8
            )
            with gr.Row():
                btn_laudo = gr.Button("📝 Gerar Laudo", variant="primary", size="lg")
                btn_criticos = gr.Button("⚠️ Achados Críticos?", variant="secondary", size="lg")

        with gr.Column(scale=1):
            laudo = gr.Markdown(label="Laudo Gerado")
            criticos = gr.Textbox(
                label="Verificação de Achados Críticos",
                lines=5,
                interactive=False
            )

    btn_laudo.click(
        analisar_completo,
        inputs=[achados, modalidade, regiao],
        outputs=[laudo, criticos]
    )
    btn_criticos.click(
        analisar_completo,
        inputs=[achados, modalidade, regiao],
        outputs=[laudo, criticos]
    )

app_v3.launch(share=True)